In [ ]:
import os
from tqdm import tqdm

import re

In [ ]:
# === Configuration ===
SERPER_API_KEY = os.getenv("SERPER_API_KEY")  # set in your environment
SERPER_API_URL = 'https://google.serper.dev/search'

Mayo 

In [2]:
import hashlib

def hash_query(query: str) -> str:
    return hashlib.md5(query.encode("utf-8")).hexdigest()

In [3]:
import requests

def search_mayo_clinic_top1_serper(query):
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    query_with_site = f"{query} site:mayoclinic.org/zh-hans"
    payload = {"q": query_with_site}

    response = requests.post(SERPER_API_URL, headers=headers, json=payload)
    if response.status_code != 200:
        raise Exception(f"Serper API error: {response.text}")

    data = response.json()
    mayo_urls = [
        item["link"] for item in data.get("organic", [])
        if "mayoclinic.org" in item["link"]
    ]

    if not mayo_urls:
        print("❌ No Mayo Clinic URL found in search results.")
        return None
    return mayo_urls[0]


In [4]:
from bs4 import BeautifulSoup

def scrape_mayo_page_text(url):
    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.content, "html.parser")
        paragraphs = soup.find_all("p")
        clean_text = "\n".join(p.get_text() for p in paragraphs if len(p.get_text()) > 40)
        return clean_text.strip()
    except Exception as e:
        print(f"⚠️ Error scraping {url}: {e}")
        return ""

In [5]:
def get_reference_knowledge_from_conversation_serper(conversation_path, save_path):
    with open(conversation_path, "r") as f:
        conv_text = f.read()

    # Extract the first User question as the query
    user_lines = re.findall(r"User: (.+)", conv_text)
    if not user_lines:
        print("⚠️ No user prompt found.")
        return ""
    query = user_lines[0]
    print(f"\n🔍 Searching Mayo Clinic for query:\n{query}\n")

    url = search_mayo_clinic_top1_serper(query)
    if not url:
        print("❌ Failed to find Mayo Clinic URL.")
        return ""

    print(f"🌐 Mayo URL: {url}")

    page_text = scrape_mayo_page_text(url)
    if not page_text:
        print("⚠️ Failed to extract page content.")
        return ""

    # save
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        f.write(page_text)

    print(f"\n📄 Saved Mayo Clinic reference to: {save_path}")
    return page_text

In [ ]:
CONVERSATION_DIR = "outputs/prompts"
REFERENCE_DIR = "outputs/rag"
os.makedirs(REFERENCE_DIR, exist_ok=True)

# Get all conversation files
conversation_files = sorted([
    f for f in os.listdir(CONVERSATION_DIR)
    if f.startswith("conversation_") and f.endswith(".txt")
])

# batch processing
for filename in tqdm(conversation_files, desc="Fetching Mayo references"):
    idx = filename.split("_")[1].split(".")[0]
    conversation_path = os.path.join(CONVERSATION_DIR, filename)
    reference_path = os.path.join(REFERENCE_DIR, f"reference_{idx}.txt")

    if os.path.exists(reference_path):
        print(f"✅ Reference already exists for conversation {idx}, skipping.")
        continue

    try:
        get_reference_knowledge_from_conversation_serper(
            conversation_path,
            reference_path
        )
    except Exception as e:
        print(f"❌ Error for conversation {idx}: {e}")
        
        

In [ ]:
import os
import re
import json
import hashlib
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ====== 配置区 ======
SERPER_API_URL = "https://google.serper.dev/search"
SERPER_API_KEY = '8372268dfeece5b866a0f382ae4a03c3dcd3d9ba'

PROBLEM_JSONL   = "data/problems_checklist_clean.jsonl"  # 你的 JSONL（含 problem 字段）
CONVERSATION_DIR = "outputs/prompts"
REFERENCE_DIR    = "outputs/rag"

os.makedirs(CONVERSATION_DIR, exist_ok=True)
os.makedirs(REFERENCE_DIR, exist_ok=True)


# ====== 工具函数 ======
def hash_query(query: str) -> str:
    return hashlib.md5(query.encode("utf-8")).hexdigest()

def search_mayo_clinic_top1_serper(query: str):
    """
    只搜中文站点，必要时可改为优先中文、找不到再回退英文：
      query_with_site = f'{query} (site:mayoclinic.org/zh-hans OR site:mayoclinic.org)'
    这里按你的要求，仅限 zh-hans。
    """
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    query_with_site = f"{query} site:mayoclinic.org/zh-hans"
    payload = {"q": query_with_site}

    resp = requests.post(SERPER_API_URL, headers=headers, json=payload, timeout=20)
    if resp.status_code != 200:
        raise Exception(f"Serper API error: {resp.text}")

    data = resp.json()
    mayo_urls = [
        item.get("link","") for item in data.get("organic", [])
        if "mayoclinic.org" in item.get("link","")
    ]
    if not mayo_urls:
        print("❌ No Mayo Clinic URL found in search results.")
        return None
    # 若想更严格只收中文，可再过滤 '/zh-hans/'：
    for u in mayo_urls:
        if "/zh-hans/" in u:
            return u
    # 如果真的想只要中文，可以 return None；这里为了容错，返回第一条
    return mayo_urls[0]

def scrape_mayo_page_text(url: str) -> str:
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        soup = BeautifulSoup(r.content, "html.parser")

        # 优先找正文区域
        main = soup.find("main") or soup
        article = main.find("article") or main

        # 抓正文段落
        parts = [t.get_text(" ", strip=True) for t in article.find_all(["h1","h2","h3","p","li"])]

        # 黑名单关键词，凡是包含这些的就不要
        blacklist = ["妙佑医疗国际", "梅奥诊所", "捐款", "联系我们", "学院", "研究与教育", "医疗专业人员"]
        clean_parts = []
        for p in parts:
            if any(bad in p for bad in blacklist):
                continue
            if len(p) < 10:  # 太短的也跳过
                continue
            clean_parts.append(p)

        return "\n".join(clean_parts).strip()
    except Exception as e:
        print(f"⚠️ Error scraping {url}: {e}")
        return ""


def get_reference_knowledge_from_conversation_serper(conversation_path: str, save_path: str):
    with open(conversation_path, "r", encoding="utf-8") as f:
        conv_text = f.read()

    # 维持你原有的“抽取第一条 User 作为查询”的逻辑
    user_lines = re.findall(r"User:\s*(.+)", conv_text)
    if not user_lines:
        print("⚠️ No user prompt found.")
        return ""
    query = user_lines[0].strip()
    print(f"\n🔍 Searching Mayo Clinic (zh-hans) for query:\n{query}\n")

    url = search_mayo_clinic_top1_serper(query)
    if not url:
        print("❌ Failed to find Mayo Clinic URL.")
        return ""

    print(f"🌐 Mayo URL: {url}")

    page_text = scrape_mayo_page_text(url)
    if not page_text:
        print("⚠️ Failed to extract page content.")
        return ""

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        # 也可以把 URL 一起写进去，便于溯源：
        f.write(url + "\n\n")
        f.write(page_text)

    print(f"\n📄 Saved Mayo Clinic reference to: {save_path}")
    return page_text

# ====== 从 JSONL 读取 problem → 写 conversation 文件 → 复用老函数 ======
rows = []
with open(PROBLEM_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        prob = obj.get("problem")
        if prob:
            rows.append(prob)

print(f"🧾 Loaded {len(rows)} problems from JSONL")

for i, problem in tqdm(list(enumerate(rows)), desc="Fetching Mayo references"):
    idx = str(i)  # 从0开始
    conversation_path = os.path.join(CONVERSATION_DIR, f"conversation_{idx}.txt")
    reference_path    = os.path.join(REFERENCE_DIR,    f"reference_{idx}.txt")

    if os.path.exists(reference_path):
        print(f"✅ Reference already exists for problem {idx}, skipping.")
        continue

    # 写 conversation
    with open(conversation_path, "w", encoding="utf-8") as wf:
        wf.write(f"User: {problem}\n")

    try:
        txt = get_reference_knowledge_from_conversation_serper(conversation_path, reference_path)
        if not txt:
            print(f"❌ No reference text for problem {idx}")
    except Exception as e:
        print(f"❌ Error for problem {idx}: {e}")



version 2

In [ ]:
import os
import re
import json
import hashlib
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# ====== 配置区 ======
SERPER_API_URL = "https://google.serper.dev/search"
SERPER_API_KEY = '8372268dfeece5b866a0f382ae4a03c3dcd3d9ba'

PROBLEM_JSONL   = "data/problems_checklist_clean.jsonl"  # 你的 JSONL（含 problem 字段）
CONVERSATION_DIR = "outputs/prompts"
REFERENCE_DIR    = "outputs/rag"

os.makedirs(CONVERSATION_DIR, exist_ok=True)
os.makedirs(REFERENCE_DIR, exist_ok=True)

from urllib.parse import urlparse, urljoin
import time

ALLOWED_PREFIXES_ZH = [
    "/zh-hans/diseases-conditions/",
    "/zh-hans/symptoms/",
    "/zh-hans/tests-procedures/",
    "/zh-hans/drugs-supplements/",
]
ALLOWED_PREFIXES_EN = [
    "/diseases-conditions/",
    "/symptoms/",
    "/tests-procedures/",
    "/drugs-supplements/",
]

# ====== 工具函数 ======
def _is_allowed_mayo_url(url: str) -> bool:
    if not url or "mayoclinic.org" not in url:
        return False
    path = urlparse(url).path or ""
    return any(path.startswith(p) for p in ALLOWED_PREFIXES_ZH + ALLOWED_PREFIXES_EN)

def search_mayo_clinic_top1_serper(query: str):
    """先 Serper（中/英都收），只保留健康资料库；失败再站内搜索"""
    headers = {"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"}
    q = f'"{query}" (site:mayoclinic.org/zh-hans OR site:mayoclinic.org)'

    payload = {"q": q, "hl": "zh-cn", "gl": "us", "num": 10}
    resp = requests.post(SERPER_API_URL, headers=headers, json=payload, timeout=20)
    if resp.status_code != 200:
        print("⚠️ Serper error:", resp.text)

    urls = []
    try:
        data = resp.json()
        for it in data.get("organic", []) or []:
            link = it.get("link", "")
            if link and _is_allowed_mayo_url(link):
                urls.append(link)
    except Exception as e:
        print("⚠️ Serper parse error:", e)

    # 调试输出前3个候选，便于核对
    if urls:
        print("候选URL(Serper):")
        for u in urls[:3]:
            print("  -", u)
        # 优先中文
        urls.sort(key=lambda u: 0 if "/zh-hans/" in u else 1)
        return urls[0]

    # —— Serper 没命中：回退到站内搜索 ——
    return _mayo_internal_search_top1(query)


def _mayo_internal_search_top1(query: str):
    """只从搜索结果列表里取第一条，避免抓到导航页"""
    url = "https://www.mayoclinic.org/zh-hans/search/search-results"
    r = requests.get(
        url,
        params={"q": query, "_": str(int(time.time()*1000))},
        headers={"User-Agent": "Mozilla/5.0", "Cache-Control": "no-cache"},
        timeout=20,
    )
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    # 结果列表标题链接（常见模板的选择器）
    a = soup.select_one("ol.search-results li a, ul.search-results li a, a.link-item__title, a.cmp-tile__title-link")
    if not a:
        return None
    href = a.get("href", "").strip()
    if not href:
        return None
    if href.startswith("/"):
        href = urljoin("https://www.mayoclinic.org", href)
    # 只接受健康资料库条目
    return href if _is_allowed_mayo_url(href) else None

def scrape_mayo_page_text(url: str) -> str:
    try:
        r = requests.get(url, timeout=20, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        soup = BeautifulSoup(r.content, "html.parser")

        # 优先找正文区域
        main = soup.find("main") or soup
        article = main.find("article") or main

        # 抓正文段落
        parts = [t.get_text(" ", strip=True) for t in article.find_all(["h1","h2","h3","p","li"])]

        # 黑名单关键词，凡是包含这些的就不要
        blacklist = ["妙佑医疗国际", "梅奥诊所", "捐款", "联系我们", "学院", "研究与教育", "医疗专业人员"]
        clean_parts = []
        for p in parts:
            if any(bad in p for bad in blacklist):
                continue
            if len(p) < 10:  # 太短的也跳过
                continue
            clean_parts.append(p)

        return "\n".join(clean_parts).strip()
    except Exception as e:
        print(f"⚠️ Error scraping {url}: {e}")
        return ""


def get_reference_knowledge_from_conversation_serper(conversation_path: str, save_path: str):
    with open(conversation_path, "r", encoding="utf-8") as f:
        conv_text = f.read()

    # 维持你原有的“抽取第一条 User 作为查询”的逻辑
    user_lines = re.findall(r"User:\s*(.+)", conv_text)
    if not user_lines:
        print("⚠️ No user prompt found.")
        return ""
    query = user_lines[0].strip()
    print(f"\n🔍 Searching Mayo Clinic (zh-hans) for query:\n{query}\n")

    url = search_mayo_clinic_top1_serper(query)
    if not url:
        print("❌ Failed to find Mayo Clinic URL.")
        return ""

    print(f"🌐 Mayo URL: {url}")

    page_text = scrape_mayo_page_text(url)
    if not page_text:
        print("⚠️ Failed to extract page content.")
        return ""

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        # 也可以把 URL 一起写进去，便于溯源：
        f.write(url + "\n\n")
        f.write(page_text)

    print(f"\n📄 Saved Mayo Clinic reference to: {save_path}")
    return page_text

# ====== 从 JSONL 读取 problem → 写 conversation 文件 → 复用老函数 ======
rows = []
with open(PROBLEM_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        prob = obj.get("problem")
        if prob:
            rows.append(prob)

print(f"🧾 Loaded {len(rows)} problems from JSONL")

for i, problem in tqdm(list(enumerate(rows)), desc="Fetching Mayo references"):
    idx = str(i)  # 从0开始
    conversation_path = os.path.join(CONVERSATION_DIR, f"conversation_{idx}.txt")
    reference_path    = os.path.join(REFERENCE_DIR,    f"reference_{idx}.txt")

    if os.path.exists(reference_path):
        print(f"✅ Reference already exists for problem {idx}, skipping.")
        continue

    # 写 conversation
    with open(conversation_path, "w", encoding="utf-8") as wf:
        wf.write(f"User: {problem}\n")

    try:
        txt = get_reference_knowledge_from_conversation_serper(conversation_path, reference_path)
        if not txt:
            print(f"❌ No reference text for problem {idx}")
    except Exception as e:
        print(f"❌ Error for problem {idx}: {e}")



单独 rerun 一个 index 并覆盖

In [11]:
def rerun_reference(idx):
    if idx < 0 or idx >= len(rows):
        print("❌ Index out of range")
        return
    problem = rows[idx]
    conversation_path = os.path.join(CONVERSATION_DIR, f"conversation_{idx}.txt")
    reference_path    = os.path.join(REFERENCE_DIR,    f"reference_{idx}.txt")

    print(f"🔄 Re-running for problem {idx}: {problem}")
    with open(conversation_path, "w", encoding="utf-8") as wf:
        wf.write(f"User: {problem}\n")

    txt = get_reference_knowledge_from_conversation_serper(conversation_path, reference_path)
    if not txt:
        print("❌ Failed to retrieve content.")


In [12]:
rerun_reference(2)

🔄 Re-running for problem 2: 那么含有皂苷的药物临床应用时应该注意什么

🔍 Searching Mayo Clinic (zh-hans) for query:
那么含有皂苷的药物临床应用时应该注意什么

🌐 Mayo URL: https://www.mayoclinic.org/zh-hans/tests-procedures/combination-birth-control-pills/in-depth/birth-control-pill/art-20045136

📄 Saved Mayo Clinic reference to: outputs/rag/reference_2.txt
